# 05 — Body Weight Prediction V1

**Objective:** given one anonymous user's measurements available through time *t*, predict the next observed body weight closest to 7 days later. A label must be observed 5–9 days after *t*; labels are never interpolated. A prediction row requires at least five prior measurements.

Leakage risks include centered/full-history rolling values, random splitting, preprocessing before splitting, and training on labels not yet observed at the validation origin. This notebook uses past/current features, fold-local sklearn preprocessing, and expanding origins whose training target dates are strictly earlier than each prediction date.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from metaform_ds.config import DataScienceConfig
from metaform_ds.modeling.weight import WeightModelConfig, evaluate_weight_model, render_weight_evaluation_report
from metaform_ds.modeling.weight.targets import build_weight_prediction_examples
from metaform_ds.utils.notebooks import load_notebook_data
settings = DataScienceConfig.from_env()
data = load_notebook_data(settings)
origin = 'SYNTHETIC' if data.mode == 'SYNTHETIC' else 'REAL'
print(f'DATA ORIGIN: {origin}')
if origin == 'SYNTHETIC': print('SYNTHETIC PIPELINE VALIDATION — these are not real-data performance results.')
else: print('REAL DATA RESULTS — read-only pseudonymized extraction.')

In [ ]:
config = WeightModelConfig()
print(f'Model: {config.model_version}')
print(f'Horizon: {config.forecast_horizon_days} days ±{config.target_tolerance_days} days')
print(f'Eligibility: {config.minimum_prior_measurements} prior measurements; minimum training rows: {config.minimum_training_rows}')
targets = build_weight_prediction_examples(data.body_weight, config)
targets.eligibility_summary

In [ ]:
print('Exclusion reasons:')
print(targets.exclusion_reasons.to_string(index=False))
if targets.examples.empty: print('Insufficient data: no observed 5–9 day targets after eligible histories.')
else: print(targets.examples[['prediction_date', 'target_date', 'current_weight_kg', 'actual_weight_kg']].head().to_string(index=False))

In [ ]:
evaluation = evaluate_weight_model(data.body_weight, data_origin=origin, config=config)
print(f'Evaluation status: {evaluation.status}; evaluated predictions: {len(evaluation.predictions)}')
print('\nMAE is the primary metric (kg). R² is omitted when mathematically uninformative.')
evaluation.comparison

In [ ]:
predictions = evaluation.predictions
if predictions.empty:
    print('Insufficient temporally valid folds for actual-versus-predicted plotting.')
else:
    view = predictions.sort_values('target_date')
    fig, ax = plt.subplots()
    ax.plot(view['target_date'], view['actual_weight_kg'], marker='o', label='Actual')
    ax.plot(view['target_date'], view['predicted_weight_kg'], marker='o', label='Linear Regression')
    ax.plot(view['target_date'], view['baseline_last_value_kg'], label='Last Value')
    ax.set(title='Observed versus predicted 7-day body weight', xlabel='Target date (UTC)', ylabel='Body weight (kg)')
    ax.legend(); fig.autofmt_xdate(); plt.show()

In [ ]:
if not predictions.empty:
    absolute_error = (predictions['actual_weight_kg'] - predictions['predicted_weight_kg']).abs()
    fig, ax = plt.subplots()
    ax.hist(absolute_error, bins=min(10, max(3, len(absolute_error))))
    ax.set(title='Linear Regression absolute prediction error', xlabel='Absolute error (kg)', ylabel='Predictions')
    plt.show()

In [ ]:
if not evaluation.comparison.empty:
    comparison = evaluation.comparison.sort_values('mae_kg')
    fig, ax = plt.subplots()
    ax.bar(comparison['model'], comparison['mae_kg'])
    ax.set(title='Temporal MAE comparison', xlabel='Method', ylabel='MAE (kg)')
    ax.tick_params(axis='x', rotation=20); plt.show()

In [ ]:
if evaluation.per_user_metrics['anonymous_user_id'].nunique() > 1:
    per_user = evaluation.per_user_metrics[evaluation.per_user_metrics['model'].eq('Linear Regression')]
    fig, ax = plt.subplots()
    ax.bar(range(len(per_user)), per_user['mae_kg'])
    ax.set(title='Linear Regression MAE by anonymous user', xlabel='Anonymous user index', ylabel='MAE (kg)')
    plt.show()
else: print('Per-user chart omitted: fewer than two evaluated users.')

In [ ]:
print(evaluation.interpretation)
if evaluation.reasons: print('Evaluation notes:', *evaluation.reasons, sep='\n- ')
print('Failure cases include sparse cadence, no target in the tolerance window, missing/invalid measurements, trend instability, and histories too short for a temporally available training fold.')
print('Production integration is not automatic: synthetic results are not real evidence, and even a real-data win needs stability, error, privacy, and product review.')
report = render_weight_evaluation_report(evaluation)
artifact_dir = settings.artifact_dir
if not artifact_dir.is_absolute() and Path.cwd().name == 'notebooks': artifact_dir = Path('..') / artifact_dir
artifact_dir.mkdir(parents=True, exist_ok=True)
report_path = artifact_dir / 'body_weight_v1_evaluation.md'
report_path.write_text(report, encoding='utf-8')
print(f'Ignored evaluation report written to: {report_path}')
print(report)